# ChatPPT Homework v0.2

**Part 1**：使用 `python-pptx` 自动生成 PowerPoint，内容涵盖**文本、图片、表格、图表**  
**Part 2**：使用 `Gradio` 搭建 ChatBot GUI，将用户输入转换为 ChatPPT Markdown 格式并生成 PPT

In [24]:
import sys
sys.path.insert(0, '../src')

---
## Part 1：python-pptx 自动生成 PowerPoint

依次演示四种幻灯片内容类型：**文本 → 图片 → 表格 → 图表**

In [42]:
import os
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.chart.data import ChartData
from pptx.enum.chart import XL_CHART_TYPE
from utils import remove_all_slides
from ppt_generator import _fix_app_xml
from template_manager import load_template, get_layout_mapping

TEMPLATE = '../templates/MasterTemplate.pptx'
OUTPUT   = '../outputs/chatppt-homework-v0.2.pptx'
os.makedirs('../outputs', exist_ok=True)

# 加载模板，读取布局映射 {布局名: index}
prs = load_template(TEMPLATE)
remove_all_slides(prs)
layout_mapping = get_layout_mapping(prs)
print('布局映射:', layout_mapping)

所有默认幻灯片已被移除。
布局映射: {'Title Only': 0, 'Title and Content': 1, 'Title and Picture 1': 2, 'Title, Content, and Picture': 3, 'Title and 2 Column 1': 4, 'Tilte and Content 1 ': 5}


### 1.1 文本幻灯片

In [26]:
# ── 幻灯片 1：标题页（Title Only）──────────────────────────────
slide = prs.slides.add_slide(prs.slide_layouts[layout_mapping['Title Only']])
slide.shapes.title.text = 'ChatPPT 自动生成演示'

# 添加副标题文本框（Title Only 布局没有内容占位符，直接插入文本框）
tb = slide.shapes.add_textbox(Inches(1.5), Inches(4.0), Inches(10), Inches(0.8))
p = tb.text_frame.paragraphs[0]
p.text = 'python-pptx · 文本 · 图片 · 表格 · 图表'
p.alignment = PP_ALIGN.CENTER
p.runs[0].font.size = Pt(20)
p.runs[0].font.color.rgb = RGBColor(0x55, 0x55, 0x55)

# ── 幻灯片 2：内容页（Title and Content）──────────────────────
slide = prs.slides.add_slide(prs.slide_layouts[layout_mapping['Title and Content']])
slide.shapes.title.text = '2024 年度业绩概述'

bullets = [
    '总收入增长 15%，达到 120 亿元',
    '市场份额扩大至 30%，位居行业第一',
    '新增用户 500 万，用户留存率提升至 85%',
    '海外业务营收占比突破 20%，同比翻倍',
]
for shape in slide.shapes:
    if shape.has_text_frame and shape != slide.shapes.title:
        tf = shape.text_frame
        tf.clear()
        for i, bullet in enumerate(bullets):
            p = tf.add_paragraph() if i > 0 else tf.paragraphs[0]
            p.text = bullet
            p.level = 0
        break

print('✓ 文本幻灯片已添加（当前共', len(prs.slides), '张）')

✓ 文本幻灯片已添加（当前共 2 张）


### 1.2 图片幻灯片

In [27]:
# ── 幻灯片 3：图片页（Title and Picture 1）──────────────────────
slide = prs.slides.add_slide(prs.slide_layouts[layout_mapping['Title and Picture 1']])
slide.shapes.title.text = '业绩图表'

IMAGE_PATH = '../images/performance_chart.png'
inserted = False

# 优先使用图片占位符（placeholder type 18 = picture）
for ph in slide.placeholders:
    if ph.placeholder_format.type == 18:
        ph.insert_picture(IMAGE_PATH)
        inserted = True
        break

# 若布局无图片占位符，直接添加图片形状
if not inserted:
    slide.shapes.add_picture(
        IMAGE_PATH,
        left=Inches(1.5), top=Inches(1.8),
        width=Inches(10), height=Inches(5.0)
    )

print('✓ 图片幻灯片已添加（当前共', len(prs.slides), '张）')

✓ 图片幻灯片已添加（当前共 3 张）


### 1.3 表格幻灯片

In [28]:
# ── 幻灯片 4：表格页（Title Only + 手动添加 table）────────────
slide = prs.slides.add_slide(prs.slide_layouts[layout_mapping['Title Only']])
slide.shapes.title.text = '各季度业绩数据'

headers = ['季度', '收入（亿元）', '同比增长', '市场份额']
rows_data = [
    ['Q1', '28.5', '+12%', '27%'],
    ['Q2', '30.2', '+14%', '28%'],
    ['Q3', '32.1', '+16%', '29%'],
    ['Q4', '29.2', '+18%', '30%'],
]

table = slide.shapes.add_table(
    rows=len(rows_data) + 1, cols=len(headers),
    left=Inches(1.0), top=Inches(1.8),
    width=Inches(11.0), height=Inches(4.5)
).table

# 表头
for col, header in enumerate(headers):
    cell = table.cell(0, col)
    cell.text = header
    run = cell.text_frame.paragraphs[0].runs[0]
    run.font.bold = True
    run.font.size = Pt(14)

# 数据行
for row_idx, row in enumerate(rows_data, start=1):
    for col_idx, val in enumerate(row):
        cell = table.cell(row_idx, col_idx)
        cell.text = val
        cell.text_frame.paragraphs[0].font.size = Pt(13)

print('✓ 表格幻灯片已添加（当前共', len(prs.slides), '张）')

✓ 表格幻灯片已添加（当前共 4 张）


### 1.4 图表幻灯片

In [29]:
# ── 幻灯片 5：图表页（Title Only + 手动添加 chart）────────────
slide = prs.slides.add_slide(prs.slide_layouts[layout_mapping['Title Only']])
slide.shapes.title.text = '季度收入趋势'

chart_data = ChartData()
chart_data.categories = ['Q1', 'Q2', 'Q3', 'Q4']
chart_data.add_series('实际收入（亿元）', (28.5, 30.2, 32.1, 29.2))
chart_data.add_series('目标收入（亿元）', (27.0, 29.0, 31.0, 33.0))

chart_frame = slide.shapes.add_chart(
    XL_CHART_TYPE.COLUMN_CLUSTERED,
    Inches(1.0), Inches(1.8),
    Inches(11.0), Inches(4.8),
    chart_data
)

chart = chart_frame.chart
chart.has_title = True
chart.chart_title.text_frame.text = '2024 年各季度收入 vs 目标'

print('✓ 图表幻灯片已添加（当前共', len(prs.slides), '张）')

✓ 图表幻灯片已添加（当前共 5 张）


### 1.5 保存并验证

In [30]:
# 保存并修正 app.xml 元数据（防止 PowerPoint 打开时报错）
prs.save(OUTPUT)
_fix_app_xml(OUTPUT, prs.slides)
print(f'✅ PPT 已保存：{OUTPUT}')

# 验证
verify = Presentation(OUTPUT)
print(f'总幻灯片数：{len(verify.slides)}')
for i, s in enumerate(verify.slides):
    title  = s.shapes.title.text if s.shapes.title else '(无标题)'
    layout = s.slide_layout.name
    # 统计非标题形状
    extras = [sh.shape_type for sh in s.shapes if sh != s.shapes.title]
    print(f'  Slide {i+1}: [{layout}] "{title}" | 额外形状类型: {extras}')

✅ PPT 已保存：../outputs/chatppt-homework-v0.2.pptx
总幻灯片数：5
  Slide 1: [Title Only] "ChatPPT 自动生成演示" | 额外形状类型: [<MSO_SHAPE_TYPE.TEXT_BOX: 17>]
  Slide 2: [Title and Content] "2024 年度业绩概述" | 额外形状类型: [<MSO_SHAPE_TYPE.PLACEHOLDER: 14>]
  Slide 3: [Title and Picture 1] "业绩图表" | 额外形状类型: [<MSO_SHAPE_TYPE.PLACEHOLDER: 14>]
  Slide 4: [Title Only] "各季度业绩数据" | 额外形状类型: [<MSO_SHAPE_TYPE.TABLE: 19>]
  Slide 5: [Title Only] "季度收入趋势" | 额外形状类型: [<MSO_SHAPE_TYPE.CHART: 3>]


---
## Part 2：Gradio ChatBot GUI

流程：**用户输入（自然语言）→ LLM（`formatter.txt` prompt）→ ChatPPT Markdown → 生成 PPTX → 下载**

In [31]:
# 加载 formatter.txt prompt
with open('../prompts/formatter.txt', encoding='utf-8') as f:
    FORMATTER_PROMPT = f.read()

print('=== Formatter Prompt ===')
print(FORMATTER_PROMPT)

=== Formatter Prompt ===
**Role**:  
You are a skilled assistant, proficient in processing written content and organizing input from various sources into a clear, standardized format that can be easily used by downstream colleagues or systems. You excel at transforming complex information into well-structured slides.

**Task**:  
Your task is to understand the user’s input and break it down into multiple slides based on the theme and logical structure. You should organize the input into slide titles and bullet points, ensuring that the content on each slide is concise and to the point. 

In Chinese 

**Format**:  
Please present the organized slide content in the following format:

```
# [Main Title]

## [Slide Title]
- [Key point 1]
- [Key point 2]

## [Slide Title]
- [Key point 1]
![Image Description](Image Path)
```


In [45]:
import re
import importlib
import openai

# 强制重新加载所有相关模块，确保使用最新代码（清除 kernel 缓存）
import data_structures, slide_builder, layout_manager, input_parser, ppt_generator
for _mod in [data_structures, slide_builder, layout_manager, input_parser, ppt_generator]:
    importlib.reload(_mod)

from layout_manager import LayoutManager
from template_manager import load_template, get_layout_mapping
from input_parser import parse_input_text
from ppt_generator import generate_presentation
from data_structures import Slide

# 确认 Slide 字段正确
assert hasattr(Slide, '__dataclass_fields__') and 'layout_id' in Slide.__dataclass_fields__, \
    f"Slide fields are wrong: {list(Slide.__dataclass_fields__.keys())}"
print('Slide fields OK:', list(Slide.__dataclass_fields__.keys()))

PPT_TEMPLATE = '../templates/MasterTemplate.pptx'


def call_llm(user_input: str, api_key: str, base_url: str = '') -> str:
    """调用 LLM，将自然语言转换为 ChatPPT Markdown 格式。"""
    client = openai.OpenAI(
        api_key=api_key,
        base_url=base_url.strip() or None,
    )
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': FORMATTER_PROMPT},
            {'role': 'user',   'content': user_input},
        ],
        temperature=0.7,
    )
    return resp.choices[0].message.content


def md_to_pptx(markdown_text: str) -> str:
    """
    将 ChatPPT Markdown 解析为 PPTX 文件。
    使用 LayoutManager 自动根据内容（有无要点/图片）分配布局。
    返回生成文件路径。
    """
    prs = load_template(PPT_TEMPLATE)
    layout_mapping = get_layout_mapping(prs)
    lm = LayoutManager(layout_mapping)
    ppt_data, title = parse_input_text(markdown_text, lm)
    safe = re.sub(r'[\\/:*?"<>|]', '_', title) or 'ChatPPT_Output'
    path = f'../outputs/{safe}.pptx'
    os.makedirs('../outputs', exist_ok=True)
    generate_presentation(ppt_data, PPT_TEMPLATE, path)
    return path


print('✓ LLM 调用与 PPTX 生成函数已定义')

Slide fields OK: ['layout_id', 'layout_name', 'content']
✓ LLM 调用与 PPTX 生成函数已定义


In [48]:
import gradio as gr
import traceback

_DEFAULT_KEY = os.environ.get('OPENAI_API_KEY', '')
_DEFAULT_URL = os.environ.get('OPENAI_BASE_URL', '')


def chat(message: str, history: list, api_key: str, base_url: str):
    if not message.strip():
        yield history, None
        return

    key = api_key.strip() or _DEFAULT_KEY
    if not key:
        yield history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': '⚠️ 请先输入 OpenAI API Key。'},
        ], None
        return

    # Step 1：LLM 格式化
    try:
        markdown = call_llm(message, key, base_url)
    except Exception as e:
        yield history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': f'❌ LLM 调用失败：{e}'},
        ], None
        return

    # Step 2：生成 PPTX
    pptx_path, status = None, ''
    try:
        pptx_path = md_to_pptx(markdown)
        status = f'✅ PPT 已生成：`{pptx_path}`'
    except Exception as e:
        status = f'⚠️ PPTX 生成失败：{e}\n\n```\n{traceback.format_exc()}\n```'

    response = (
        f'{status}\n\n'
        f'**ChatPPT Markdown：**\n\n'
        f'```markdown\n{markdown}\n```'
    )
    yield history + [
        {'role': 'user',      'content': message},
        {'role': 'assistant', 'content': response},
    ], pptx_path


with gr.Blocks(title='ChatPPT') as demo:
    gr.Markdown('# 🎯 ChatPPT — AI 驱动的 PPT 生成器')
    gr.Markdown(
        '输入任意主题或内容描述，AI 将自动格式化为 ChatPPT Markdown 并生成 PowerPoint 文件。'
    )

    with gr.Row():
        api_key_box = gr.Textbox(
            label='OpenAI API Key',
            placeholder='sk-...',
            type='password',
            value=_DEFAULT_KEY,
            scale=3,
        )
        base_url_box = gr.Textbox(
            label='API Base URL（可选，留空使用默认）',
            placeholder='https://api.openai.com/v1',
            value=_DEFAULT_URL,
            scale=3,
        )

    chatbot = gr.Chatbot(
        label='对话记录',
        height=420,
    )

    with gr.Row():
        msg_box = gr.Textbox(
            label='输入 PPT 主题或内容',
            placeholder='例如：用 6 页 PPT 介绍人工智能在医疗领域的应用，包括技术原理、应用场景和挑战',
            lines=3,
            scale=5,
        )
        with gr.Column(scale=1, min_width=120):
            submit_btn = gr.Button('🚀 生成 PPT', variant='primary')
            clear_btn  = gr.Button('🗑️ 清空对话')

    file_output = gr.File(label='📥 下载 PPT 文件')

    submit_btn.click(
        fn=chat,
        inputs=[msg_box, chatbot, api_key_box, base_url_box],
        outputs=[chatbot, file_output],
    )
    msg_box.submit(
        fn=chat,
        inputs=[msg_box, chatbot, api_key_box, base_url_box],
        outputs=[chatbot, file_output],
    )
    clear_btn.click(lambda: ([], None), outputs=[chatbot, file_output])

demo.launch(share=False, allowed_paths=['../outputs'])


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


所有默认幻灯片已被移除。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: OpenClaw 介绍
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: 什么是 OpenClaw
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: OpenClaw 是一个开源的网络爬虫框架。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 旨在帮助开发者快速构建和部署网络爬虫应用。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: OpenClaw 的特点
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 高度可扩展性，支持多种爬虫策略。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 内置简单易用的 API，便于集成和使用。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 支持分布式爬取，提高抓取效率。
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: OpenClaw 的应用场景
2026-04-01 15:29:13 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 数据采集与分析：可用于市场调研、竞争对手分析等。
2026-04-01 15:29